# Stage 4. Image Background Correction Pipeline

### 1.3 Load configuration

`CONFIG_FILE = None` uses the packaged defaults. Relative paths start at the repository root.

`prepare_output(config)` saves `config.resolved.yaml` alongside the results. This is an automatic record of the effective settings; do not edit it. Later stages check the same record to avoid mixing different settings. If settings change, select another experiment/output folder. It is not an input-data file.


## 1. Setup

### 1.1 Imports

In [ ]:
# Built-in imports
import logging

# Third-party imports
from pathlib import Path

# Package imports
from acid.image_processing.background.load_background_function import (
    load_background_function,
)
from acid.utils.filesystem.filesystem import create_output_directories
from acid.utils.metadata.loading import load_metadata
from acid.utils.metadata.filtering import filter_metadata_by_splits
from acid.utils.fov_axis_utils import get_fov_ch_shape


### 1.2 Configure logging

In [ ]:
logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s - %(levelname)s - %(message)s",
)

### 1.3 Load configuration

`CONFIG_FILE = None` uses the packaged defaults. Relative paths start at the repository root.

`prepare_output(config)` saves `config.resolved.yaml` alongside the results. This is an automatic record of the effective settings; do not edit it. Later stages check the same record to avoid mixing different settings. If settings change, select another experiment/output folder. It is not an input-data file.


In [ ]:
from acid.config import load_config, prepare_output

# Use the same file in every stage. Set to None for defaults only.
CONFIG_FILE = None  # "configs/exp_001.yaml"
config = load_config(CONFIG_FILE)
# Save effective settings, or reject changes to an existing output folder.
prepare_output(config)

In [ ]:
paths_cfg = config.shared.paths

# Metadata config
metadata_cfg = config.background_correction.metadata

# Dataset split config
dataset_split_cfg = config.background_correction.dataset_split

# Background function config
background_function_cfg = config.background_correction.background_function_selection

# Background processing params. config
background_processing_cfg = config.background_correction.processing

## 2. Input Data Preparation

### 2.1. Create output directories

In [ ]:
for input_directory in (
    metadata_cfg.directory,
    paths_cfg.extracted_fov_dir,
    paths_cfg.background_functions_dir,
):
    if not Path(input_directory).is_dir():
        raise FileNotFoundError(
            f"Required input directory does not exist: {input_directory}"
        )

output_path, _ = create_output_directories(
    output_directory=paths_cfg.corrected_fov_dir,
    enable_secondary_output=False,
)

### 2.2 Load metadata dataframe

In [ ]:
metadata_df, metadata_file_name = load_metadata(metadata_config=metadata_cfg)

### 2.3 Select training images

In [ ]:
metadata_df = filter_metadata_by_splits(
    metadata_df=metadata_df, selected_splits="train", split_config=dataset_split_cfg
)

In [ ]:
metadata_df.head(3)

### 2.4 Load background data

In [ ]:
backgrounds, background_file_names = load_background_function(
    background_config=background_function_cfg,
    metadata_df=metadata_df,
)

In [ ]:
background_shapes = (
    {key: value.shape for key, value in backgrounds.items()}
    if isinstance(backgrounds, dict)
    else backgrounds.shape
)
logging.info("Background(s) shape: %s", background_shapes)
logging.info(f"Background(s) filenames: {background_file_names}")

### 2.5 Get the shape and the number of channels of the fields of view

In [ ]:
# get the shape of the individual fields of view, the number of channels and the shape of individual channels
fov_shape, num_channels, shape_of_channels = get_fov_ch_shape(
    df=metadata_df,
    fov_dir=paths_cfg.extracted_fov_dir,
    fov_clm=metadata_cfg.dataframe_columns.fov_column_name,
    channel_axis=background_processing_cfg.channel_axis,
    null_value=metadata_cfg.dataframe_columns.null_value,
)
if not isinstance(fov_shape, tuple):
    raise FileNotFoundError(f"No readable FOVs found in {paths_cfg.extracted_fov_dir}")

## 3. Background Correction

### 3.1 Pipeline functions

The background-correction functions live in `acid.image_processing.background.apply_background_correction`.


In [ ]:
from acid.image_processing.background.apply_background_correction import (
    apply_background_correction_batch,
    update_metadata_with_correction_results,
)
from acid.utils.metadata.saving import save_metadata_dataframe

In [ ]:
getattr(config, "background_correction").metadata

### 3.2 Apply background correction in batch

In [ ]:
results = apply_background_correction_batch(
    metadata_df=metadata_df,
    backgrounds=backgrounds,
    config=config.background_correction,
    paths=paths_cfg,
    max_rows=None,
)

In [ ]:
results

### 3.3 Update Processing Metadata dataframe

In [ ]:
metadata_df_updated = update_metadata_with_correction_results(
    metadata_df=metadata_df,
    results=results,
    dataframe_columns=metadata_cfg.dataframe_columns,
    copy_dataframe=True,
)

In [ ]:
metadata_df_updated.info()

In [ ]:
metadata_df.info()

### 3.4 Save metadata file

In [ ]:
config.background_correction.metadata

In [ ]:
metadata_path = save_metadata_dataframe(
    metadata_df=metadata_df_updated,
    metadata_config=metadata_cfg,
    project_name=config.project_identity.project_name,
)


logging.info(f"Saved metadata dataframe to: {metadata_path}")

## 4. Display results

In [ ]:
import random
import os
from datetime import datetime

import matplotlib.pyplot as plt
import tifffile

from acid.image_visualization.display_illumination_correction import (
    plot_image_channels_and_diagonals,
)

In [ ]:
background_correction_cfg = config.background_correction
project_identity_cfg = config.project_identity
metadata_config = background_correction_cfg.metadata
metadata_columns_cfg = metadata_config.dataframe_columns
paths_cfg = config.shared.paths
processing_cfg = background_correction_cfg.processing
image_saving_cfg = background_correction_cfg.image_saving
graphs_saving_cfg = background_correction_cfg.graphs_saving

In [ ]:
project_name = project_identity_cfg.project_name

fov_column_name = metadata_columns_cfg.fov_column_name
illum_correct_df_file_name_clm_name = (
    metadata_columns_cfg.illum_correct_df_file_name_clm_name
)

fov_directory = paths_cfg.extracted_fov_dir
output_directory = image_saving_cfg.directory
channel_axis = processing_cfg.channel_axis

save_graphs = graphs_saving_cfg.save_graphs
graph_saving_directory = graphs_saving_cfg.graph_saving_directory
graph_date_format = graphs_saving_cfg.graph_date_format
graph_suffix = graphs_saving_cfg.graph_suffix
graph_savingword = graphs_saving_cfg.graph_savingword
save_file_name_separator = image_saving_cfg.save_file_name_separator

In [ ]:
# randomly select a corrected field of view before and after correction
metadata_display_df = metadata_df_updated[
    metadata_df_updated[illum_correct_df_file_name_clm_name].notna()
]

random_field_of_view_idx = random.choice(list(metadata_display_df.index))
random_field_of_view_name = metadata_display_df.loc[
    random_field_of_view_idx, fov_column_name
]
random_field_of_view_corrected_name = metadata_display_df.loc[
    random_field_of_view_idx, illum_correct_df_file_name_clm_name
]
logging.info(f"""
displaying the result for:
{random_field_of_view_name}
{random_field_of_view_corrected_name}
""")
random_field_of_view = tifffile.imread(
    os.path.join(fov_directory, random_field_of_view_name)
)
random_field_of_view_corrected = tifffile.imread(
    os.path.join(output_directory, random_field_of_view_corrected_name)
)


fig, axes = plt.subplots(num_channels, 4, figsize=(12, 3 * num_channels))

plot_image_channels_and_diagonals(
    random_field_of_view,
    random_field_of_view_corrected,
    channel_axis=channel_axis,
    axes=axes,
    title_1="Original",
    title_2="Corrected",
    title_3="Main Diagonal",
    title_4="Anti Diagonal",
    x_axis="Pixel index",
    y_axis="Intensity",
    normalize=False,
)


# save graph
if save_graphs:
    graph_saving_name = f"{datetime.now().strftime(graph_date_format)}{save_file_name_separator}{project_name}{save_file_name_separator}{graph_savingword}{graph_suffix}"
    os.makedirs(graph_saving_directory, exist_ok=True)
    fig.savefig(os.path.join(graph_saving_directory, graph_saving_name))
    logging.info(f"graph saved as {graph_saving_name} in {graph_saving_directory}")
else:
    logging.info("graph not saved, set save_graphs to True to save the graph")

# End of the notebook